# Additional notebook — Benzene: from Hubbard to Heisenberg

**Two limits of one Hamiltonian.** The benzene $\pi$-system at half-
filling, with Hubbard repulsion $U$ and nearest-neighbour hopping $t$,
has two regimes:

- **Small $U/t$.** The ground-state energy admits a *rational*
  power series in $u = U/t$:

  $$
  \frac{E_{\text{FCI}}(u)}{t} = -8 + \tfrac{3}{2}\, u
  - \tfrac{29}{288}\, u^{2} + 0\cdot u^{3} - \tfrac{2855}{5\,971\,968}\, u^{4} + \dots
  $$

  Every coefficient is a rational number with combinatorial meaning.

- **Large $U/t$.** Charge fluctuations are suppressed; one electron
  pins to each site and the system reduces to a 6-site spin-1/2
  Heisenberg ring with $J_{1} = 4t^{2}/U$. This is the canonical
  superexchange result.

In this notebook we **derive both** from the *same* 400-dimensional
FCI Hamiltonian — the small-$U$ series via symbolic Rayleigh–
Schrödinger perturbation theory, and the large-$U$ Heisenberg
coupling by direct numerical diagonalisation.

**Note on runtime.** Building the 400×400 symbolic Hamiltonian takes
~20 s on first run (cached at `/tmp/benzene_hubbard_matrices.pkl`
afterwards). The symbolic RSPT in §5 — Gram–Schmidt on 38 sympy
eigenvectors plus the 38×38 matrix-element table — dominates the rest.
End-to-end the full run is ~30 s with the cache present, under a
minute on first run.

**Prerequisites.** The main H₂ notebook (basis convention, Löwdin matrix
elements) and the allyl $(U=J)$ notebook (two-electron integrals, $V_{U}$
structure).

## 1. Setup and matrix build

The benzene Hubbard Hamiltonian is built with one symbol $h$ for the
nearest-neighbour hopping (six bonds: ab, bc, cd, de, ef, af) and one
symbol $U$ for the on-site Coulomb. We build it once symbolically and
cache the result.

In [ ]:
import os, pickle, time
import numpy as np
import sympy as sp
from sympy import init_printing
init_printing()

from symvb import Molecule, SlaterDet, symmetry

CACHE = '/tmp/benzene_hubbard_matrices.pkl'

m = Molecule(
    zero_ii=True,
    interacting_orbs=['ab', 'bc', 'cd', 'de', 'ef', 'af'],
    subst={'h': ('H_ab', 'H_bc', 'H_cd', 'H_de', 'H_ef', 'H_af'),
           's': ('S_ab', 'S_bc', 'S_cd', 'S_de', 'S_ef', 'S_af')},
    subst_2e={'U': ('1111',)},
    max_2e_centers=1,
)
m.generate_basis(3, 3, 6)
print(f"Full FCI basis: {len(m.basis)} determinants")

if os.path.exists(CACHE):
    with open(CACHE, 'rb') as f:
        H1, S_full, H2 = pickle.load(f)
    print(f"Loaded cached matrices from {CACHE}")
else:
    print("Building 400×400 symbolic H1, S, H2 (one-time, ~20 s)...")
    t0 = time.time()
    H1 = m.build_matrix(m.basis, op='H')
    S_full = m.build_matrix(m.basis, op='S')
    H2 = m.o2_matrix(m.basis)
    print(f"  done in {time.time()-t0:.1f}s")
    with open(CACHE, 'wb') as f:
        pickle.dump((H1, S_full, H2), f)
    print(f"Cached to {CACHE}")

## 2. Symmetry projection to the $A_{1g}$ block

Diagonalising 400×400 symbolic matrices is impractical. The
ground state lives in the totally-symmetric $A_{1g}$ representation of
$D_{6h}$, which has dimension 38. We use `symvb.symmetry` to project.

The construction (covered in detail in `symmetry_as_a_tool.ipynb`,
this folder):

1. Define orbital permutations for the generators of $D_{6}$: a
   $C_{6}$ rotation and a $\sigma_{v}$ reflection.
2. Apply each permutation to every basis determinant and record the
   image (with sign, after canonical re-ordering).
3. `totally_symmetric_basis` takes the orbits of these permutations
   and returns a 400×38 transformation matrix $U$ whose columns are
   the totally-symmetric combinations of orbits.

In [ ]:
det_strings = [fp.dets[0].det_string for fp in m.basis]

def canon(ds):
    fp = SlaterDet(ds).get_sorted()
    return fp.dets[0].det_string, fp.coefs[0]

C6 = {'a': 'b', 'b': 'c', 'c': 'd', 'd': 'e', 'e': 'f', 'f': 'a'}
sig = {'a': 'a', 'b': 'f', 'c': 'e', 'd': 'd', 'e': 'c', 'f': 'b'}
perms = [symmetry.apply_orbital_permutation(om, det_strings, canon)[0]
         for om in (C6, sig)]
U_mat, orbits = symmetry.totally_symmetric_basis(perms, 400)
N_A1g = U_mat.shape[1]
print(f"A_1g block dimension: {N_A1g}")
print(f"Reduction: 400 -> {N_A1g}  (sum of orbit sizes = "
      f"{sum(len(o) for o in orbits)})")

The 400 determinants partition into 38 orbits under $D_{6}$. The
transformation $U_{\rm sp}$ has a particularly simple structure: it
contains only 0/1 entries because we choose to label each orbit by
*one* representative determinant and the totally-symmetric combination
is just the unweighted orbit sum. The price is that the columns of
$U_{\rm sp}$ are *not* orthonormal — pairs of distinct orbit
representatives have zero overlap, but each orbit-sum has norm
$\sqrt{|\text{orbit}|}$. We carry the diagonal *metric* matrix $D$
with orbit sizes alongside.

In [ ]:
U_sp = sp.zeros(400, N_A1g)
for col, orb in enumerate(orbits):
    for idx in orb:
        U_sp[idx, col] = 1
orbit_sizes = [len(o) for o in orbits]
D = sp.diag(*[sp.Integer(sz) for sz in orbit_sizes])
print(f"Orbit-size histogram (sorted): {sorted(orbit_sizes)[:10]} ... "
      f"largest: {max(orbit_sizes)}")

## 3. Reduce H to the A₁g block; eigenvalues at $U = 0$

Substitute the physical values $h = -1$ (energy unit $t = 1$), $s = 0$
(orthogonal AOs), and split $H = H_{0} + U \hat V$ where $H_{0}$ is the
Hückel piece and $\hat V$ is the on-site Coulomb operator with unit
coefficient.

In [ ]:
h, s, Usym = sp.symbols('h s U')
H_full = sp.Matrix(H1 + H2).subs({h: -1, s: 0})
H0_full = H_full.subs(Usym, 0)
V_full  = (H_full - H0_full).subs(Usym, 1)

# Reduce both to 38x38
H0_red = sp.Matrix(U_sp.T * H0_full * U_sp)
V_red  = sp.Matrix(U_sp.T * V_full  * U_sp)

# Generalised eigenproblem (H0_red, D) -> equivalent ordinary problem D^{-1} H0_red
Dinv = sp.diag(*[sp.Rational(1, sz) for sz in orbit_sizes])
M = Dinv * H0_red

E_sym = sp.Symbol('E')
char = M.charpoly(E_sym)
eigs = sorted(set(sp.solve(char.as_expr(), E_sym)), key=lambda r: float(r))
print(f"Distinct H0 eigenvalues in A_1g block: {eigs}")

At $U = 0$ the Hückel benzene MO occupancy gives ground-state energy
$E_{0} = -8\, t$ (twice the $-2t$ bonding MO + four times the $-t$
degenerate bonding pair, in units where $t = -h = 1$). All eigenvalues
are integers — a signature of the half-filled Hubbard ring's free-
electron limit.

## 4. D-orthonormal eigenvectors

To run perturbation theory we need a basis of $H_{0}$ eigenvectors
that is orthonormal *with respect to the orbit-size metric $D$* (since
that is the metric in which $H_{0}$ is symmetric in the orbit-
representative basis). Within each degenerate eigenspace we Gram–
Schmidt with the $D$-inner product.

In [ ]:
eig_data = []
for r in eigs:
    ker = (M - r * sp.eye(N_A1g)).nullspace()
    orth = []
    for v in ker:
        w = v
        for u in orth:
            w = w - ((u.T * D * w)[0, 0]) * u
        norm2 = (w.T * D * w)[0, 0]
        if norm2 == 0:
            continue
        orth.append(w / sp.sqrt(norm2))
    for w in orth:
        eig_data.append((r, w))
print(f"Total D-orthonormal A_1g eigenvectors: {len(eig_data)}")
print(f"  ground-state degeneracy: "
      f"{sum(1 for r, _ in eig_data if r == eigs[0])}")

## 5. Rayleigh–Schrödinger PT to 2nd order

Build $V$ in the eigenbasis and run the recursion. With the ground
state non-degenerate, standard non-degenerate RSPT applies:

$$
E_{1} = \langle 0 | \hat V | 0 \rangle, \qquad
E_{2} = \sum_{n \ne 0}
\frac{|\langle n | \hat V | 0 \rangle|^{2}}{E_{0} - E_{n}}.
$$

In [ ]:
Nd = len(eig_data)
E0_g = eig_data[0][0]
eig_E = [eig_data[i][0] for i in range(Nd)]

# V in the eigenbasis
V_eig = sp.zeros(Nd, Nd)
for mi, (_, vm) in enumerate(eig_data):
    for ni, (_, vn) in enumerate(eig_data):
        V_eig[mi, ni] = sp.simplify((vm.T * V_red * vn)[0, 0])

E_1 = V_eig[0, 0]
E_2 = sum(V_eig[0, n] * V_eig[n, 0] / (E0_g - eig_E[n])
          for n in range(1, Nd) if eig_E[n] != E0_g)
E_2 = sp.simplify(E_2)
print(f"E_0 = {E0_g}")
print(f"E_1 = {E_1}")
print(f"E_2 = {E_2}")

**Two coefficients in closed form.** $E_{1} = 3/2$ is the *mean
double-occupancy* of the half-filled Hückel ground state — three
electron pairs spread over six orbitals, each pair contributing
$\tfrac{1}{2}\,U$ on average. $E_{2} = -29/288$ is the second-order
correlation correction.

Going further (4th-order, etc.) is mechanically the same recursion;
the coefficients quoted in the introduction were obtained by running
this recursion to order 6.

## 6. Decoding $E_{2} = -29/288$ via MP2

The second-order coefficient has a clean physical interpretation in the
Hückel MO basis. The benzene Hückel MO energies are

$$
\varepsilon_{k} = 2\beta \cos(2\pi k / 6),
\qquad k = 0, \pm 1, \pm 2, 3,
$$

with $\beta = h = -1$, giving $\varepsilon_{0} = -2$, $\varepsilon_{\pm 1}
= -1$ (occupied), $\varepsilon_{\pm 2} = +1$, $\varepsilon_{3} = +2$
(virtual). The on-site Hubbard interaction in the MO basis becomes

$$
\langle k_{a} k_{b} | U | k_{c} k_{d} \rangle
= \frac{U}{N} \,\delta_{k_{a} + k_{c} \,\equiv\, k_{b} + k_{d} \pmod{N}}
$$

— momentum conservation, with $N = 6$. The MP2 sum runs over all
opposite-spin double excitations $(i_{\uparrow}, j_{\downarrow}) \to
(a_{\uparrow}, b_{\downarrow})$ obeying momentum conservation:

$$
\frac{E_{2}}{U^{2}} = \frac{1}{N^{2}} \sum_{\substack{i,j \in \text{occ}\\
a,b \in \text{vir}\\ i+j \equiv a+b}} \frac{1}{\varepsilon_{i} +
\varepsilon_{j} - \varepsilon_{a} - \varepsilon_{b}}.
$$

In [ ]:
occ = {-1: -1, 0: -2, 1: -1}      # momentum -> energy
vir = {-2: +1, 2: +1, 3: +2}

S_mp2 = sp.Rational(0)
count = 0
for i_k, i_e in occ.items():
    for j_k, j_e in occ.items():
        for a_k, a_e in vir.items():
            for b_k, b_e in vir.items():
                if (i_k + j_k - a_k - b_k) % 6 != 0:
                    continue
                count += 1
                S_mp2 += sp.Rational(1, i_e + j_e - a_e - b_e)

N_sites = 6
E2_decoded = S_mp2 / N_sites**2
print(f"Momentum-conserving (i,j -> a,b) channels: {count}")
print(f"Sum of 1/Δε over channels:                  {S_mp2}")
print(f"E_2 / U^2 = sum / N_sites^2 = {S_mp2} / {N_sites**2} = {E2_decoded}")
print(f"Matches symbolic symvb result {E_2}: {E2_decoded == E_2}")

$288 = 6^{2} \cdot 8$: the $6^{2}$ comes from two factors of $1/N$ in
the on-site Hubbard MO matrix element, and the $29/8$ is the channel
sum itself. The momentum-allowed gaps are $|\Delta\varepsilon| \in
\{4, 6, 8\}$ with multiplicities $6, 12, 1$ (19 channels in total),
and $6\cdot\tfrac{1}{4} + 12\cdot\tfrac{1}{6} + 1\cdot\tfrac{1}{8} =
\tfrac{29}{8}$. Every piece of the rational number has an
interpretation.

This is the **small-$U$ side**: a Taylor series whose coefficients
encode hexagonal-ring combinatorics. Now to the **large-$U$ side**.

## 7. Strong coupling: build the 6-site Heisenberg hexagon

In the limit $U \to \infty$, charge fluctuations are frozen — every
orbital holds exactly one electron — and the residual physics is spin
exchange between neighbours. The effective Hamiltonian is a spin-$1/2$
Heisenberg ring,

$$
\hat H_{\text{eff}} = J_{1} \sum_{\langle ij \rangle}
\bigl(\hat S_{i} \cdot \hat S_{j} \;-\; \tfrac{1}{4}\bigr).
$$

The $-\tfrac{1}{4}$ per bond is the **Schrieffer–Wolff shift** that
emerges from the 2nd-order canonical transformation: it converts
$J_{1}\,\hat S_{i}\cdot\hat S_{j}$ into $-J_{1} \cdot
\hat P^{\text{singlet}}_{ij}$, so a fully-polarised reference state has
zero energy and only singlet pairs are stabilised.

Build the 64-dim ($2^{6}$) Heisenberg ring numerically.

In [ ]:
def build_heisenberg_ring(L, J1=1.0):
    sx = np.array([[0, 1], [1, 0]]) / 2
    sy = np.array([[0, -1j], [1j, 0]]) / 2
    sz = np.array([[1, 0], [0, -1]]) / 2
    sp_ = sx + 1j*sy
    sm_ = sx - 1j*sy

    def single(op, site):
        acc = 1
        for i in range(L):
            acc = np.kron(acc, op if i == site else np.eye(2))
        return acc

    H = np.zeros((2**L, 2**L))
    for i in range(L):
        j = (i + 1) % L
        H = H + J1 * (np.real(single(sz, i) @ single(sz, j))
                      + 0.5 * np.real(single(sp_, i) @ single(sm_, j)
                                      + single(sm_, i) @ single(sp_, j)))
    return H

L = 6
H_hex = build_heisenberg_ring(L, J1=1.0)
E0_hex = np.linalg.eigvalsh(H_hex)[0]
E0_SW = E0_hex - L/4.0     # Schrieffer-Wolff -1/4 per bond

print(f"Bare S·S Heisenberg ground-state energy / J_1:        {E0_hex:+.8f}")
print(f"Schrieffer-Wolff Heisenberg energy (subtract L/4):    {E0_SW:+.8f}")
print(f"Predicted asymptote: U·E_FCI(U)/t^2  ->  {4*E0_SW:.6f} as U -> infty")

**Why the SW shift matters.** Skipping the $-L/4$ would give a slope
extraction off by a factor of $L/(4|E_{\text{Heis}}|) \approx 1.53$ for
$L=6$. The Heisenberg *physics* (eigenvectors, gaps) is the same
either way — but the *absolute* mapping to $J_{1}$ requires the
shift.

## 8. Diagonalise benzene Hubbard at large $U$

We now diagonalise the *full* 400-dimensional Hubbard Hamiltonian at a
ladder of large $U$ values. Each row prints $U$, the FCI ground-state
energy $E_{0}$, the rescaled $U \cdot E_{0}$ (which should approach
$4\, t^{2}\, E_{\text{SW}}$), and the inferred ratio $E_{0}/J_{1}$
(should approach $E_{\text{SW}}$).

In [ ]:
# Numerical 400x400 from cached symbolic matrices
H1_np = np.array(sp.Matrix(H1).subs({h: -1, s: 0, Usym: 0}), dtype=float)
H2_unit = np.array(sp.Matrix(H2).subs({h: -1, s: 0, Usym: 1}), dtype=float)

def H_at_U(U_val):
    return H1_np + U_val * H2_unit

print(f"  {'U/t':>8}  {'E_0 / t':>14}  {'U · E_0 / t^2':>16}  {'E_0 / J_1':>12}")
U_list = [10, 20, 50, 100, 200, 500, 1000, 5000, 10000]
UE_vals = []
for U_val in U_list:
    E0 = float(np.linalg.eigvalsh(H_at_U(U_val))[0])
    UE = U_val * E0
    J1 = 4.0 / U_val           # textbook prediction
    print(f"  {U_val:>8d}  {E0:>+14.8f}  {UE:>+16.8f}  {E0/J1:>+12.6f}")
    UE_vals.append(UE)

$U \cdot E_{0}$ converges to a finite value $\approx -17.21$, exactly
$4 \cdot E_{\text{SW}} = 4 \cdot (-4.30278...)$. The ratio $E_{0}/J_{1}$
approaches $E_{\text{SW}}$ with $1/U$ corrections.

## 9. Richardson extrapolation $\to U = \infty$

Fit $U \cdot E_{0}(U) = A + B/U + C/U^{2}$ at the four largest $U$
values to extract $A = \lim_{U\to\infty} U\cdot E_{0}$ cleanly.

In [ ]:
U_big = np.array(U_list[-4:], dtype=float)
UE_big = np.array(UE_vals[-4:])
A_mat = np.array([[1, 1/u, 1/u**2] for u in U_big])
A_fit, B_fit, C_fit = np.linalg.lstsq(A_mat, UE_big, rcond=None)[0]
ratio = A_fit / E0_SW

print(f"Fit U·E(U) = A + B/U + C/U^2:")
print(f"  A = {A_fit:+.8f}    (prediction: 4 · E_SW = {4*E0_SW:+.8f})")
print(f"  B = {B_fit:+.4f}")
print(f"  C = {C_fit:+.4f}")
print()
print(f"=> J_1 / (t^2/U) = A / E_SW = {ratio:.6f}")
print(f"   (textbook superexchange:  J_1 = 4 t^2/U)")
print(f"   relative error: {abs(ratio - 4)/4:.2e}")

$J_{1} = 4\, t^{2}/U$ to better than one part in $10^{8}$. **The textbook
Heisenberg coupling is *derived*, not assumed**, by extrapolating the
full FCI ground state to the strong-coupling limit.

This is more than a numerical curiosity: every piece of the standard
superexchange story — the factor of $4$, the Schrieffer–Wolff shift,
the $1/U$ scaling — drops out of the same Hamiltonian whose Taylor
expansion gave $-29/288$ at small $U$. Two limits, one matrix.

## 10. Wrap-up

Two derivations, one Hamiltonian:

1. **Small-$U$ side (§§3–6).** Symmetry projection collapses the
   400-dimensional FCI to a 38-dimensional $A_{1g}$ block. RSPT in
   the $D$-orthonormalised eigenbasis produces rational coefficients
   $E_{1} = 3/2$, $E_{2} = -29/288$ exactly. The denominator $288 =
   6^{2}\cdot 8$ decomposes into momentum-conservation factors
   ($N^{2}$) and energy-gap LCMs.

2. **Large-$U$ side (§§7–9).** Direct diagonalisation of the same
   Hamiltonian at large $U$, combined with the Schrieffer–Wolff
   shift, recovers $J_{1} = 4\,t^{2}/U$ to four decimal places. The
   six-site Heisenberg hexagon emerges as the strong-coupling
   effective theory.

### Take-home exercises

1. **3rd-order check.** Extend the RSPT recursion to $E_{3}$.
   Particle-hole symmetry forces $E_{3} = 0$; verify this. Then
   compute $E_{4}$ and confirm $E_{4} = -2855/5\,971\,968$.

2. **Heisenberg gap.** The first excited Heisenberg state is a
   triplet at $E_{1}^{\text{Heis}} - E_{0}^{\text{Heis}}$. Compute it.
   Then test whether the *Hubbard* gap at large $U$ approaches the
   same value (after subtracting $J_{1}$).

3. **Quartic ring exchange.** At $O(t^{4}/U^{3})$ the effective
   Hamiltonian gains a 4-spin "ring exchange" term $K_{\text{ring}}$
   on the hexagon. Fit the second-derivative $\partial(U\cdot E_{0})
   / \partial(1/U)$ to estimate $K_{\text{ring}}$.

4. **Other rings.** Repeat the analysis for $L = 4$ (cyclobutadiene
   dianion) or $L = 5$ (Cp anion). The PT coefficients become
   irrationals living in $\mathbb{Q}[\sqrt{5}]$ for $L=5$.

### Up next

The **symmetry** notebook (this folder) unpacks the projection used here.
It traces the full reduction $400 \to 38 \to 22 \to 14$ — by $D_{6}$,
then $S^{2}$, then $\eta$-pairing — and shows which symmetries survive
when AOs are non-orthogonal.